In [1]:
import keras
import pandas as pd
import numpy as np
import math
import matplotlib as plt
import os
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

2026-08-04 17:57:52.378741: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-04 17:57:53.701369: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-04 17:57:56.955828: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
np.set_printoptions(threshold=np.inf)

In [3]:
#####DATA FORMATTING#####

train = pd.read_csv("/home/gr00vingm0nkey/vscode/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/home/gr00vingm0nkey/vscode/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

print(train.head())

print(test.head())

numeric_feats = train.select_dtypes(include=[np.number]).drop(
    columns=["Id", "SalePrice", "SalePrice_log"], errors="ignore").columns

print(numeric_feats)

def Data(train, test):
    train = train.copy()
    test = test.copy()

    # Save target
    y_train = np.log1p(train.pop("SalePrice").astype("int32"))

    # Remove IDs
    train.drop(columns=["Id"], inplace=True)
    test.drop(columns=["Id"], inplace=True)

    # Combine so train and test get identical columns
    combined = pd.concat([train, test], axis=0, ignore_index=True)

    # Fill missing numeric values
    numeric_cols = combined.select_dtypes(include=["number"]).columns
    combined[numeric_cols] = combined[numeric_cols].fillna(
        combined[numeric_cols].median()
    )
    min_max = MinMaxScaler()
    combined[numeric_cols] = np.round(min_max.fit_transform(combined[numeric_cols]),4)

    # Fill missing categorical values
    categorical_cols = combined.select_dtypes(include=["object"]).columns
    combined[categorical_cols] = combined[categorical_cols].fillna("Missing")

    # One-hot encode
    combined = pd.get_dummies(combined, dtype="int32")

    # Ensure all columns are int32
    combined = combined.astype("float32")

    # Split back while RETAINING column names
    x_train = combined.iloc[:len(train)].reset_index(drop=True)
    x_test = combined.iloc[len(train):].reset_index(drop=True)

    return x_train, x_test, y_train


x_train, x_test, y_train = Data(train, test)


print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(x_train.head)
# print(x_test)



   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  
0   2008        WD   

In [4]:
#####NORMAL_MODEL######

@keras.utils.register_keras_serializable(package="custom_layers")
class Normal_Model(keras.Model):
    def __init__(self, units, **kwargs):
        super(Normal_Model, self).__init__(**kwargs)
        # print("CM__init__----")

        self.layer1 = keras.layers.Dense(units=2048,activation=keras.activations.leaky_relu)
        self.layer2 = keras.layers.Dense(units=1024, activation=keras.activations.leaky_relu)
        self.layer3 = keras.layers.Dense(units=512, activation=keras.activations.leaky_relu)
        self.final = keras.layers.Dense(units=1, activation=keras.activations.linear)
        pass

    def build(self, input_shape):
        # print("CMbuild----")
        #super.build()
        pass

    def call(self, inputs):
        # print("CMcall----")
        x1 = self.layer1(inputs)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.final(x3)
        return x4

In [5]:
#####GA_MODEL######

@keras.utils.register_keras_serializable(package="custom_layers")
class Genetic_Algorithm(keras.layers.Layer):
    def __init__(self, name=None, kernel_initializer=None, **kwargs):
        print("CLK__init__----")
        super().__init__(**kwargs)

        self.population_size = 40
        self.population = None
        self.x_train, _, self.y_train = Data(train, test)

    def build(self, input_shape):
        print("CLKbuild----")
        print(input_shape)

        num_features = input_shape[-1]

        self.population = tf.Variable(
            tf.random.normal((self.population_size, num_features)),
            trainable=False,
            dtype=tf.float32,
        )

        super().build(input_shape)

    def call(self, inputs, model):
        rinputs, kept = self.Reduce(self.x_train)
        ry_train = tf.gather(self.y_train, kept)
        losses = self.Evaluate_Genes(self.population, model, rinputs, ry_train)
        new_population, best_gene = self.Evolve(self.population, losses)
        self.population.assign(new_population)
        return inputs * best_gene

    def Reduce(self, arr):
        n = tf.shape(arr)[0]
        keep_n = n - n // 4
        scores = tf.random.uniform([n])
        keep_indices = tf.math.top_k(scores, k=keep_n).indices
        keep_indices = tf.sort(keep_indices)

        return tf.gather(arr, keep_indices), keep_indices

    def Evaluate_Genes(self, population, model, inputs, y_train):
        loss_fn = tf.keras.losses.MeanSquaredError(reduction=tf.keras.losses.Reduction.NONE)
        x = tf.expand_dims(inputs, axis=0)
        genes = tf.expand_dims(population, axis=1)
        for layer in model:
            if layer == "GA":
                x = x * genes
            else:
                x = layer(x)
        if x.shape.ndims == 3 and x.shape[-1] == 1:
            x = tf.squeeze(x, axis=-1)

        y_true = tf.reshape(tf.cast(y_train, x.dtype), [1, -1])
        y_true = tf.broadcast_to(y_true, tf.shape(x))

        Eval = loss_fn(y_true, x)
        return Eval

    def Evolve(self, population, fitness):
        pop_size = tf.shape(population)[0]
        num_features = tf.shape(population)[1]
        elite_size = pop_size // 4
        num_children = elite_size * 2

        # --- selection ---
        order = tf.argsort(fitness, direction="DESCENDING")
        elites = tf.gather(population, order[:elite_size])
        Best_gene = elites[0]

        # --- crossover: vectorized, no Python loop ---
        i_idx = tf.random.uniform([num_children], 0, elite_size, dtype=tf.int32)
        j_idx = tf.random.uniform([num_children], 0, elite_size, dtype=tf.int32)

        p1 = tf.gather(elites, i_idx)
        p2 = tf.gather(elites, j_idx)

        mask = tf.random.uniform(tf.shape(p1)) < 0.5
        children = tf.where(mask, p1, p2)

        # --- mutation ---
        mutant_idx = tf.random.uniform([elite_size], 0, elite_size, dtype=tf.int32)
        mutants = tf.gather(elites, mutant_idx)
        scale = tf.minimum(fitness[0], 1.0) * 0.2
        mutants += tf.random.uniform(tf.shape(mutants), -scale, scale)  

        return tf.concat([elites, children, mutants], axis=0), Best_gene

@keras.utils.register_keras_serializable(package="custom_layers")
class GA_Model(keras.Model):
    def __init__(self, units, **kwargs):
        super(GA_Model, self).__init__(**kwargs)
        print("CM__init__----")
        self.GA = Genetic_Algorithm(name="GA")
        self.layer1 = keras.layers.Dense(units=2048,activation=keras.activations.leaky_relu)
        self.layer2 = keras.layers.Dense(units=1024, activation=keras.activations.leaky_relu)
        self.layer3 = keras.layers.Dense(units=512, activation=keras.activations.leaky_relu)
        self.final = keras.layers.Dense(units=1, activation=keras.activations.linear)
        pass

    def build(self, input_shape):
        print("CMbuild----")
        #super.build()
        pass

    def call(self, inputs):
        print("CMcall----")
        x1 = self.layer1(inputs)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.GA(x3, model=[self.layer1, self.layer2, self.layer3, "GA", self.final])
        x5 = self.final(x4)
        return x5

In [6]:
#####FUNCTIONS######

def custom_loss(y_true, y_pred):

    mse = tf.reduce_mean(tf.square(y_true - y_pred))

    error = y_true - y_pred

    percent_error = tf.where(y_true != 0, (y_pred - y_true) / y_true, tf.zeros_like(y_true))

    error = mse + error + (mse+error*percent_error)
    return error

def unscaled_rmse(y_true, y_pred):
    y_true_orig = tf.math.expm1(y_true)
    y_pred_orig = tf.math.expm1(y_pred)
    return tf.sqrt(tf.reduce_mean(tf.square(y_true_orig - y_pred_orig)))

def Plot_Loss(Data, Title, split):
    n_groups = max(split) + 1
    fig, axes = plt.pyplot.subplots(n_groups, 1, figsize=(8, 4 * n_groups))
    if n_groups == 1:
        axes = [axes]

    for z in range(len(Data)):
        ax = axes[split[z]]
        ploty = Data[z]
        skip  = len(ploty) // 5
        ploty = ploty[skip:]
        plotx = np.arange(skip, skip + len(ploty))
        ax.plot(plotx, ploty, label=Title[z])

    for ax in axes:
        ax.yaxis.set_major_locator(plt.ticker.MaxNLocator(10))
        ax.legend(loc="best", fontsize=9)

    fig.tight_layout()
    plt.pyplot.savefig("plot.png", bbox_inches="tight")
    plt.pyplot.show()

def create_submission(model, x_test, test_ids, filename="submission.csv"):
    """Predict on x_test, invert the log1p target transform, and write a
    Kaggle-style submission CSV with columns [Id, SalePrice]."""
    preds_log = model.predict(x_test)
    preds = np.expm1(preds_log).flatten()

    submission = pd.DataFrame({
        "Id": test_ids,
        "SalePrice": preds
    })

    submission.to_csv(filename, index=False)
    print(f"Saved {len(submission)} rows to {filename}")
    return submission

In [7]:
ES = keras.callbacks.EarlyStopping(monitor='loss',patience=3, restore_best_weights=True)
# Normmodel = Normal_Model(2)
GAmodel = GA_Model(2)

GAmodel.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001), 
    loss=custom_loss(),
    metrics=[unscaled_rmse]
)
# Normmodel.compile(
    # optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    # loss=keras.losses.MeanSquaredError(),
    # metrics=[unscaled_rmse]
# )

EPOCHS = 120

GAhistory = GAmodel.fit(x_train, y_train, epochs=EPOCHS, verbose=2, validation_split=0.15)
# Normhistory = Normmodel.fit(x_train, y_train, epochs=EPOCHS, verbose=2, validation_split=0.15)



# Plot_Loss([
#     GAhistory.history["loss"],
#     GAhistory.history["unscaled_rmse"],
#     GAhistory.history["val_loss"],
#     GAhistory.history["val_unscaled_rmse"],
#     Normhistory.history["loss"],
#     Normhistory.history["unscaled_rmse"],
#     Normhistory.history["val_loss"],
#     Normhistory.history["val_unscaled_rmse"]
# ], [
#     "GAloss",
#     "GAunscaled_rmse",
#     "GAval_loss",
#     "GAval_unscaled_rmse",
#     "Normloss",
#     "Normunscaled_rmse",
#     "Normval_loss",
#     "Normval_unscaled_rmse"
# ], [
#     0,
#     1,
#     2,
#     3,
#     0,
#     1,
#     2,
#     3
# ])

test_ids = test["Id"]
submission = create_submission(GAmodel, x_test, test_ids, "housingGAsubmission.csv")

########TODO########### WHEN GETTING PREDICTIONS ON TEST, np.expm1 IT#############TODO#########

CM__init__----
CLK__init__----


I0000 00:00:1785884283.593469  677559 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


TypeError: custom_loss() missing 2 required positional arguments: 'y_true' and 'y_pred'